<a href="https://colab.research.google.com/github/g-aditya1828/DS-Lab-2P11/blob/main/10ai_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score,silhouette_samples
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy import stats
import warnings
import os
import glob
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

class EnergyAnalysisPipeline:
    """Complete pipeline for energy consumption analysis with accuracy metrics"""

    def __init__(self):
        self.scaler = StandardScaler()
        self.pca = None
        self.kmeans = None
        self.feature_names = None
        self.accuracy_metrics = {}

    # ========== CUSTOM DATASET INPUT ==========

    def get_dataset_path(self):

            data_path = input("\nEnter directory path containing CSV files: ").strip()
            if not os.path.exists(data_path):
                print(f"⚠ Warning: Path '{data_path}' does not exist. Using default.")
                return '/mnt/user-data/uploads/', 'default'
            print(f"✓ Using custom directory: {data_path}")
            return data_path, 'custom_dir'



    # ========== PHASE 1: DATA LOADING & PREPARATION ==========

    def load_and_merge_data(self, data_source=None, source_type='default'):
        """Load all block files and merge with household information"""
        print("\n" + "=" * 70)
        print("PHASE 1: DATA LOADING & PREPARATION")
        print("=" * 70)

        if data_source is None:
            data_source, source_type = self.get_dataset_path()

        blocks = []

        if source_type == 'default' or source_type == 'custom_dir':
            # Load block files
            data_path = data_source if isinstance(data_source, str) else data_source[0]

            # Try to find block files
            block_files = glob.glob(os.path.join(data_path, 'block_*.csv'))

            if not block_files:
                print(f"⚠ No block_*.csv files found in {data_path}")
                # Try to load any CSV files
                csv_files = glob.glob(os.path.join(data_path, '*.csv'))
                csv_files = [f for f in csv_files if 'information' not in f.lower()
                            and 'weather' not in f.lower() and 'holiday' not in f.lower()]
                print(f"Found {len(csv_files)} CSV files to process...")
                block_files = csv_files[:10]  # Limit to first 10 files

            for i, block_file in enumerate(sorted(block_files)):
                try:
                    df = pd.read_csv(block_file)
                    blocks.append(df)
                    print(f"✓ Loaded {os.path.basename(block_file)}: {len(df)} records")
                except Exception as e:
                    print(f"✗ Error loading {block_file}: {e}")

            # Load household information if available
            info_files = glob.glob(os.path.join(data_path, '*information*.csv'))
            households = None
            if info_files:
                try:
                    households = pd.read_csv(info_files[0])
                    print(f"✓ Household metadata: {len(households)} buildings")
                except:
                    print("⚠ Could not load household information")

            # Load weather data if available
            weather_files = glob.glob(os.path.join(data_path, '*weather*.csv'))
            weather = None
            if weather_files:
                try:
                    weather = pd.read_csv(weather_files[0])
                    if 'time' in weather.columns:
                        weather['time'] = pd.to_datetime(weather['time'])
                        weather['date'] = weather['time'].dt.date
                    print(f"✓ Weather data: {len(weather)} records")
                except:
                    print("⚠ Could not load weather data")

        elif source_type == 'custom_files':
            for file_path in data_source:
                try:
                    df = pd.read_csv(file_path)
                    blocks.append(df)
                    print(f"✓ Loaded {os.path.basename(file_path)}: {len(df)} records")
                except Exception as e:
                    print(f"✗ Error loading {file_path}: {e}")

            households = None
            weather = None

        if not blocks:
            raise ValueError("No data files were successfully loaded!")

        energy_data = pd.concat(blocks, ignore_index=True)
        print(f"\n✓ Total energy records: {len(energy_data)}")

        # Try to identify building ID column
        id_columns = ['LCLid', 'building_id', 'id', 'meter_id', 'customer_id']
        building_id_col = None
        for col in id_columns:
            if col in energy_data.columns:
                building_id_col = col
                break

        if building_id_col:
            print(f"✓ Unique buildings ({building_id_col}): {energy_data[building_id_col].nunique()}")
        else:
            print("⚠ Warning: Could not identify building ID column")
            print(f"Available columns: {list(energy_data.columns)}")

        return energy_data, households, weather

    # ========== PHASE 2: FEATURE ENGINEERING ==========

    def engineer_features(self, energy_data, weather=None):
        """Create comprehensive features for each building"""
        print("\n" + "=" * 70)
        print("PHASE 2: FEATURE ENGINEERING")
        print("=" * 70)

        # Detect building ID column
        id_columns = ['LCLid', 'building_id', 'id', 'meter_id', 'customer_id']
        building_id_col = None
        for col in id_columns:
            if col in energy_data.columns:
                building_id_col = col
                break

        if not building_id_col:
            print("⚠ Creating synthetic building IDs...")
            energy_data['building_id'] = range(len(energy_data))
            building_id_col = 'building_id'

        features_list = []

        for building_id in energy_data[building_id_col].unique():
            building_data = energy_data[energy_data[building_id_col] == building_id].copy()

            if len(building_data) < 10:  # Skip buildings with insufficient data
                continue

            # Detect date column
            date_col = None
            for col in ['day', 'date', 'timestamp', 'time']:
                if col in building_data.columns:
                    date_col = col
                    break

            if date_col:
                try:
                    building_data[date_col] = pd.to_datetime(building_data[date_col])
                    building_data['weekday'] = building_data[date_col].dt.dayofweek
                    building_data['month'] = building_data[date_col].dt.month
                except:
                    pass

            features = {building_id_col: building_id}

            # Detect energy columns
            energy_cols = [col for col in building_data.columns if 'energy' in col.lower()
                          or 'consumption' in col.lower() or 'kwh' in col.lower()]

            # === 1. PEAK DEMAND FEATURES ===
            if 'energy_max' in building_data.columns:
                features['max_demand'] = building_data['energy_max'].max()
            elif energy_cols:
                features['max_demand'] = building_data[energy_cols[0]].max()
            else:
                features['max_demand'] = 0

            if 'energy_mean' in building_data.columns:
                features['mean_consumption'] = building_data['energy_mean'].mean()
            elif energy_cols:
                features['mean_consumption'] = building_data[energy_cols[0]].mean()
            else:
                features['mean_consumption'] = 0

            features['peak_to_avg_ratio'] = features['max_demand'] / (features['mean_consumption'] + 1e-10)
            features['peak_frequency'] = (building_data.get('energy_max', pd.Series([0])).fillna(0) >
                                         building_data.get('energy_max', pd.Series([0])).fillna(0).quantile(0.9)).sum() / len(building_data)

            # === 2. LOAD VARIABILITY FEATURES ===
            features['std_consumption'] = building_data.get('energy_mean', building_data[energy_cols[0]] if energy_cols else pd.Series([0])).std()
            features['coefficient_of_variation'] = features['std_consumption'] / (features['mean_consumption'] + 1e-10)
            features['load_factor'] = features['mean_consumption'] / (features['max_demand'] + 1e-10)

            if 'energy_min' in building_data.columns:
                features['energy_range'] = features['max_demand'] - building_data['energy_min'].min()
            else:
                features['energy_range'] = features['max_demand']

            # === 3. TIME-BASED FEATURES ===
            if 'weekday' in building_data.columns:
                weekday_data = building_data[building_data['weekday'] < 5]
                weekend_data = building_data[building_data['weekday'] >= 5]

                weekday_mean = weekday_data.get('energy_mean', weekday_data[energy_cols[0]] if energy_cols else pd.Series([0])).mean() if len(weekday_data) > 0 else 0
                weekend_mean = weekend_data.get('energy_mean', weekend_data[energy_cols[0]] if energy_cols else pd.Series([0])).mean() if len(weekend_data) > 0 else 0
                features['weekday_weekend_ratio'] = weekday_mean / (weekend_mean + 1e-10)
            else:
                features['weekday_weekend_ratio'] = 1.0

            # Seasonal patterns
            if 'month' in building_data.columns:
                winter_months = building_data[building_data['month'].isin([12, 1, 2])]
                summer_months = building_data[building_data['month'].isin([6, 7, 8])]

                winter_mean = winter_months.get('energy_mean', winter_months[energy_cols[0]] if energy_cols else pd.Series([0])).mean() if len(winter_months) > 0 else 0
                summer_mean = summer_months.get('energy_mean', summer_months[energy_cols[0]] if energy_cols else pd.Series([0])).mean() if len(summer_months) > 0 else 0
                features['winter_summer_ratio'] = winter_mean / (summer_mean + 1e-10)
            else:
                features['winter_summer_ratio'] = 1.0

            # === 4. DERIVED METRICS ===
            features['base_load'] = building_data.get('energy_min', pd.Series([0])).quantile(0.1) if 'energy_min' in building_data.columns else 0
            features['total_consumption'] = building_data.get('energy_sum', building_data[energy_cols[0]] if energy_cols else pd.Series([0])).sum()
            features['avg_daily_consumption'] = features['total_consumption'] / max(len(building_data), 1)
            features['consumption_volatility'] = building_data.get('energy_sum', building_data[energy_cols[0]] if energy_cols else pd.Series([0])).std()

            # === 5. STATISTICAL METRICS ===
            features['median_consumption'] = building_data.get('energy_median', building_data.get('energy_mean', building_data[energy_cols[0]] if energy_cols else pd.Series([0]))).median()
            features['skewness'] = building_data.get('energy_mean', building_data[energy_cols[0]] if energy_cols else pd.Series([0])).skew()
            features['kurtosis'] = building_data.get('energy_mean', building_data[energy_cols[0]] if energy_cols else pd.Series([0])).kurtosis()

            features_list.append(features)

        features_df = pd.DataFrame(features_list)
        print(f"\n✓ Engineered {len(features_df.columns) - 1} features for {len(features_df)} buildings")
        print(f"\nFeature Categories:")
        print("  - Peak Demand Features: 4")
        print("  - Load Variability Features: 4")
        print("  - Time-Based Features: 2")
        print("  - Derived Metrics: 4")
        print("  - Statistical Metrics: 3")

        return features_df

    # ========== PHASE 3: DATA PREPROCESSING ==========

    def preprocess_data(self, features_df):
        """Clean, handle outliers, and scale features"""
        print("\n" + "=" * 70)
        print("PHASE 3: DATA PREPROCESSING")
        print("=" * 70)

        # Separate ID from features
        id_columns = ['LCLid', 'building_id', 'id', 'meter_id', 'customer_id']
        id_col = None
        for col in id_columns:
            if col in features_df.columns:
                id_col = col
                break

        building_ids = features_df[id_col] if id_col else pd.Series(range(len(features_df)))
        X = features_df.drop(id_col, axis=1) if id_col else features_df

        # Handle missing values
        missing_before = X.isnull().sum().sum()
        X = X.fillna(X.median())
        print(f"✓ Handled {missing_before} missing values")

        # Handle infinite values
        X = X.replace([np.inf, -np.inf], np.nan)
        X = X.fillna(X.median())

        # Remove outliers using IQR method
        Q1 = X.quantile(0.25)
        Q3 = X.quantile(0.75)
        IQR = Q3 - Q1
        outlier_mask = ~((X < (Q1 - 3 * IQR)) | (X > (Q3 + 3 * IQR))).any(axis=1)

        X_cleaned = X[outlier_mask]
        building_ids_cleaned = building_ids[outlier_mask]
        print(f"✓ Removed {len(X) - len(X_cleaned)} outlier buildings")

        # Feature correlation analysis
        corr_matrix = X_cleaned.corr()
        high_corr = np.where(np.abs(corr_matrix) > 0.95)
        high_corr_pairs = [(corr_matrix.index[x], corr_matrix.columns[y])
                           for x, y in zip(*high_corr) if x != y and x < y]

        if high_corr_pairs:
            print(f"✓ Found {len(high_corr_pairs)} highly correlated feature pairs (>0.95)")

        # Standardization
        X_scaled = self.scaler.fit_transform(X_cleaned)
        self.feature_names = X_cleaned.columns.tolist()
        print(f"✓ Standardized {len(self.feature_names)} features (mean=0, std=1)")

        return X_scaled, building_ids_cleaned, X_cleaned

    # ========== PHASE 4: PCA ANALYSIS ==========

    def perform_pca(self, X_scaled, n_components=None):
        """Apply PCA for dimensionality reduction"""
        print("\n" + "=" * 70)
        print("PHASE 4: PCA DIMENSIONALITY REDUCTION")
        print("=" * 70)

        # Initial PCA to determine optimal components
        pca_full = PCA()
        pca_full.fit(X_scaled)

        # Determine components for 90% variance
        cumsum_variance = np.cumsum(pca_full.explained_variance_ratio_)
        n_components_90 = np.argmax(cumsum_variance >= 0.90) + 1
        n_components_95 = np.argmax(cumsum_variance >= 0.95) + 1

        print(f"\n✓ Components for 90% variance: {n_components_90}")
        print(f"✓ Components for 95% variance: {n_components_95}")

        # Use optimal components
        if n_components is None:
            n_components = n_components_90

        self.pca = PCA(n_components=n_components)
        X_pca = self.pca.fit_transform(X_scaled)

        print(f"\n✓ Reduced from {X_scaled.shape[1]} to {n_components} dimensions")
        print(f"✓ Explained variance: {self.pca.explained_variance_ratio_.sum():.2%}")

        # Store PCA accuracy metrics
        self.accuracy_metrics['pca_variance_retained'] = self.pca.explained_variance_ratio_.sum()
        self.accuracy_metrics['pca_n_components'] = n_components
        self.accuracy_metrics['pca_reconstruction_error'] = np.mean((X_scaled - self.pca.inverse_transform(X_pca))**2)

        # Component interpretation
        print("\n" + "-" * 70)
        print("TOP 3 FEATURES PER PRINCIPAL COMPONENT:")
        print("-" * 70)

        components_df = pd.DataFrame(
            self.pca.components_,
            columns=self.feature_names,
            index=[f'PC{i+1}' for i in range(n_components)]
        )

        for i in range(min(3, n_components)):
            pc = f'PC{i+1}'
            top_features = components_df.loc[pc].abs().nlargest(3)
            print(f"\n{pc} ({self.pca.explained_variance_ratio_[i]:.1%} variance):")
            for feature, loading in top_features.items():
                sign = '+' if components_df.loc[pc, feature] > 0 else '-'
                print(f"  {sign} {feature}: {abs(loading):.3f}")

        return X_pca, components_df

    # ========== PHASE 5: K-MEANS CLUSTERING ==========

    def determine_optimal_clusters(self, X_pca, max_k=10):
        """Find optimal number of clusters using multiple methods"""
        print("\n" + "=" * 70)
        print("PHASE 5: DETERMINING OPTIMAL CLUSTERS")
        print("=" * 70)

        inertias = []
        silhouette_scores = []
        davies_bouldin_scores = []
        calinski_harabasz_scores = []
        k_range = range(2, max_k + 1)

        for k in k_range:
            kmeans = KMeans(n_clusters=k, n_init=50, max_iter=300, random_state=42)
            labels = kmeans.fit_predict(X_pca)

            inertias.append(kmeans.inertia_)
            silhouette_scores.append(silhouette_score(X_pca, labels))
            davies_bouldin_scores.append(davies_bouldin_score(X_pca, labels))
            calinski_harabasz_scores.append(calinski_harabasz_score(X_pca, labels))

        # Find optimal k
        diffs = np.diff(inertias)
        elbow_k = np.argmax(diffs[:-1] - diffs[1:]) + 3  # +3 due to starting at k=2
        silhouette_k = k_range[np.argmax(silhouette_scores)]
        db_k = k_range[np.argmin(davies_bouldin_scores)]
        ch_k = k_range[np.argmax(calinski_harabasz_scores)]

        print(f"\n✓ Elbow Method suggests: {elbow_k} clusters")
        print(f"✓ Best Silhouette Score at: {silhouette_k} clusters ({max(silhouette_scores):.3f})")
        print(f"✓ Best Davies-Bouldin at: {db_k} clusters ({min(davies_bouldin_scores):.3f})")
        print(f"✓ Best Calinski-Harabasz at: {ch_k} clusters ({max(calinski_harabasz_scores):.1f})")

        # Use majority vote or best silhouette
        optimal_k = silhouette_k
        print(f"\n>>> RECOMMENDED: {optimal_k} clusters")

        return optimal_k, {
            'inertias': inertias,
            'silhouette': silhouette_scores,
            'davies_bouldin': davies_bouldin_scores,
            'calinski_harabasz': calinski_harabasz_scores,
            'k_range': list(k_range)
        }

    def apply_kmeans(self, X_pca, n_clusters):
        """Apply K-Means clustering with comprehensive accuracy metrics"""
        print("\n" + "=" * 70)
        print("PHASE 6: K-MEANS CLUSTERING")
        print("=" * 70)

        self.kmeans = KMeans(
            n_clusters=n_clusters,
            n_init=100,
            max_iter=500,
            random_state=42
        )

        cluster_labels = self.kmeans.fit_predict(X_pca)

        # ========== COMPREHENSIVE ACCURACY METRICS ==========

        # 1. Silhouette Score (-1 to 1, higher is better)
        silhouette = silhouette_score(X_pca, cluster_labels)

        # 2. Davies-Bouldin Index (lower is better)
        davies_bouldin = davies_bouldin_score(X_pca, cluster_labels)

        # 3. Calinski-Harabasz Score (higher is better)
        calinski_harabasz = calinski_harabasz_score(X_pca, cluster_labels)

        # 4. Inertia (within-cluster sum of squares)
        inertia = self.kmeans.inertia_

        # 5. Cluster balance metrics
        unique, counts = np.unique(cluster_labels, return_counts=True)
        cluster_balance = counts.std() / counts.mean()  # Lower is more balanced

        # Store metrics
        self.accuracy_metrics.update({
            'n_clusters': n_clusters,
            'silhouette_score': silhouette,
            'davies_bouldin_index': davies_bouldin,
            'calinski_harabasz_score': calinski_harabasz,
            'inertia': inertia,
            'cluster_balance_cv': cluster_balance,
            'cluster_sizes': dict(zip(unique, counts))
        })

        # Print validation metrics
        print(f"\n{'='*70}")
        print("CLUSTERING ACCURACY METRICS")
        print(f"{'='*70}")
        print(f"\n✓ Clustered {len(cluster_labels)} buildings into {n_clusters} groups")
        print(f"\n📊 QUALITY METRICS:")
        print(f"   • Silhouette Score: {silhouette:.4f}")
        print(f"     Range: -1 to 1 | Current: {'Excellent' if silhouette > 0.5 else 'Good' if silhouette > 0.3 else 'Acceptable' if silhouette > 0.2 else 'Poor'}")
        print(f"     Interpretation: Measures how well-separated clusters are")
        print(f"\n   • Davies-Bouldin Index: {davies_bouldin:.4f}")
        print(f"     Range: 0 to ∞ (lower is better) | Current: {'Excellent' if davies_bouldin < 1 else 'Good' if davies_bouldin < 1.5 else 'Acceptable'}")
        print(f"     Interpretation: Ratio of within-cluster to between-cluster distances")
        print(f"\n   • Calinski-Harabasz Score: {calinski_harabasz:.2f}")
        print(f"     Range: 0 to ∞ (higher is better) | Current: {'Excellent' if calinski_harabasz > 1000 else 'Good' if calinski_harabasz > 500 else 'Acceptable'}")
        print(f"     Interpretation: Ratio of between-cluster to within-cluster variance")
        print(f"\n   • Inertia (WCSS): {inertia:.2f}")
        print(f"     Interpretation: Sum of squared distances to nearest cluster center (lower is better)")
        print(f"\n   • Cluster Balance (CV): {cluster_balance:.4f}")
        print(f"     Interpretation: Coefficient of variation of cluster sizes (lower = more balanced)")

        # Per-cluster silhouette scores
        print(f"\n📈 PER-CLUSTER QUALITY:")
        # Calculate silhouette samples once for the entire dataset
        sample_silhouette_values = silhouette_samples(X_pca, cluster_labels)

        for i in unique:
            cluster_mask = cluster_labels == i
            # Calculate the mean silhouette score for samples belonging to cluster i
            cluster_silhouette = sample_silhouette_values[cluster_mask].mean()
            print(f"   Cluster {i}: Silhouette = {cluster_silhouette:.4f} ({counts[i]} buildings)")

        # Cluster sizes
        print(f"\n📊 CLUSTER DISTRIBUTION:")
        for cluster_id, count in zip(unique, counts):
            percentage = (count / len(cluster_labels)) * 100
            bar = '█' * int(percentage / 2)
            print(f"   Cluster {cluster_id}: {bar} {count:>4} buildings ({percentage:>5.1f}%)")

        return cluster_labels

    # ========== ACCURACY REPORT ==========

    def print_accuracy_report(self):
        """Print comprehensive accuracy and performance report"""
        print("\n" + "=" * 70)
        print("COMPREHENSIVE ACCURACY & PERFORMANCE REPORT")
        print("=" * 70)

        print("\n" + "─" * 70)
        print("1. DIMENSIONALITY REDUCTION (PCA) PERFORMANCE")
        print("─" * 70)
        print(f"   Original Dimensions: {len(self.feature_names)}")
        print(f"   Reduced Dimensions: {self.accuracy_metrics['pca_n_components']}")
        print(f"   Dimension Reduction: {(1 - self.accuracy_metrics['pca_n_components']/len(self.feature_names))*100:.1f}%")
        print(f"   Variance Retained: {self.accuracy_metrics['pca_variance_retained']*100:.2f}%")
        print(f"   Information Loss: {(1-self.accuracy_metrics['pca_variance_retained'])*100:.2f}%")
        print(f"   Reconstruction Error: {self.accuracy_metrics['pca_reconstruction_error']:.6f}")
        print(f"   Efficiency Score: {'★★★★★' if self.accuracy_metrics['pca_variance_retained'] > 0.95 else '★★★★☆' if self.accuracy_metrics['pca_variance_retained'] > 0.90 else '★★★☆☆'}")

        print("\n" + "─" * 70)
        print("2. CLUSTERING QUALITY METRICS")
        print("─" * 70)
        print(f"   Number of Clusters: {self.accuracy_metrics['n_clusters']}")
        print(f"\n   A. Silhouette Score: {self.accuracy_metrics['silhouette_score']:.4f}")
        print(f"      • Interpretation: {self._interpret_silhouette(self.accuracy_metrics['silhouette_score'])}")
        print(f"      • Quality Rating: {self._rate_metric(self.accuracy_metrics['silhouette_score'], 'silhouette')}")

        print(f"\n   B. Davies-Bouldin Index: {self.accuracy_metrics['davies_bouldin_index']:.4f}")
        print(f"      • Interpretation: {self._interpret_davies_bouldin(self.accuracy_metrics['davies_bouldin_index'])}")
        print(f"      • Quality Rating: {self._rate_metric(self.accuracy_metrics['davies_bouldin_index'], 'db')}")

        print(f"\n   C. Calinski-Harabasz Score: {self.accuracy_metrics['calinski_harabasz_score']:.2f}")
        print(f"      • Interpretation: {self._interpret_calinski_harabasz(self.accuracy_metrics['calinski_harabasz_score'])}")
        print(f"      • Quality Rating: {self._rate_metric(self.accuracy_metrics['calinski_harabasz_score'], 'ch')}")

        print(f"\n   D. Cluster Balance: {self.accuracy_metrics['cluster_balance_cv']:.4f}")
        print(f"      • Interpretation: {self._interpret_balance(self.accuracy_metrics['cluster_balance_cv'])}")

        print("\n" + "─" * 70)
        print("3. CLUSTER SIZE DISTRIBUTION")
        print("─" * 70)
        for cluster_id, size in self.accuracy_metrics['cluster_sizes'].items():
            total = sum(self.accuracy_metrics['cluster_sizes'].values())
            pct = (size / total) * 100
            print(f"   Cluster {cluster_id}: {size:>4} buildings ({pct:>5.1f}%)")

        print("\n" + "─" * 70)
        print("4. OVERALL MODEL QUALITY ASSESSMENT")
        print("─" * 70)

        # Calculate overall quality score
        quality_score = self._calculate_overall_quality()

        print(f"   Overall Quality Score: {quality_score:.2f}/100")
        print(f"   Rating: {self._overall_rating(quality_score)}")
        print(f"   Recommendation: {self._overall_recommendation(quality_score)}")

        print("\n" + "=" * 70)

    def _interpret_silhouette(self, score):
        if score > 0.7:
            return "Strong cluster structure, well-separated clusters"
        elif score > 0.5:
            return "Good cluster structure, clusters are reasonably separated"
        elif score > 0.3:
            return "Moderate cluster structure, some overlap between clusters"
        elif score > 0.2:
            return "Weak cluster structure, significant overlap"
        else:
            return "Poor cluster structure, clusters may not be meaningful"

    def _interpret_davies_bouldin(self, score):
        if score < 0.5:
            return "Excellent cluster separation"
        elif score < 1.0:
            return "Good cluster separation"
        elif score < 1.5:
            return "Moderate cluster separation"
        else:
            return "Poor cluster separation, clusters overlap"

    def _interpret_calinski_harabasz(self, score):
        if score > 1000:
            return "Excellent between-cluster variance"
        elif score > 500:
            return "Good between-cluster variance"
        elif score > 200:
            return "Moderate between-cluster variance"
        else:
            return "Low between-cluster variance"

    def _interpret_balance(self, cv):
        if cv < 0.2:
            return "Very balanced cluster sizes"
        elif cv < 0.4:
            return "Reasonably balanced cluster sizes"
        elif cv < 0.6:
            return "Somewhat imbalanced cluster sizes"
        else:
            return "Highly imbalanced cluster sizes"

    def _rate_metric(self, value, metric_type):
        if metric_type == 'silhouette':
            if value > 0.7: return "★★★★★ (Excellent)"
            elif value > 0.5: return "★★★★☆ (Very Good)"
            elif value > 0.3: return "★★★☆☆ (Good)"
            elif value > 0.2: return "★★☆☆☆ (Fair)"
            else: return "★☆☆☆☆ (Poor)"

        elif metric_type == 'db':
            if value < 0.5: return "★★★★★ (Excellent)"
            elif value < 1.0: return "★★★★☆ (Very Good)"
            elif value < 1.5: return "★★★☆☆ (Good)"
            elif value < 2.0: return "★★☆☆☆ (Fair)"
            else: return "★☆☆☆☆ (Poor)"

        elif metric_type == 'ch':
            if value > 1000: return "★★★★★ (Excellent)"
            elif value > 500: return "★★★★☆ (Very Good)"
            elif value > 200: return "★★★☆☆ (Good)"
            elif value > 100: return "★★☆☆☆ (Fair)"
            else: return "★☆☆☆☆ (Poor)"

    def _calculate_overall_quality(self):
        # Weighted quality score
        silhouette_score = min(self.accuracy_metrics['silhouette_score'] / 0.7, 1.0) * 30
        db_score = max(1 - self.accuracy_metrics['davies_bouldin_index'] / 2.0, 0) * 30
        ch_score = min(self.accuracy_metrics['calinski_harabasz_score'] / 1000, 1.0) * 20
        pca_score = self.accuracy_metrics['pca_variance_retained'] * 20

        return silhouette_score + db_score + ch_score + pca_score

    def _overall_rating(self, score):
        if score >= 85: return "⭐⭐⭐⭐⭐ EXCELLENT"
        elif score >= 70: return "⭐⭐⭐⭐ VERY GOOD"
        elif score >= 55: return "⭐⭐⭐ GOOD"
        elif score >= 40: return "⭐⭐ FAIR"
        else: return "⭐ NEEDS IMPROVEMENT"

    def _overall_recommendation(self, score):
        if score >= 85:
            return "Model is production-ready with high confidence in cluster assignments."
        elif score >= 70:
            return "Model performs well and is suitable for deployment with monitoring."
        elif score >= 55:
            return "Model is acceptable but consider refining features or trying different k values."
        elif score >= 40:
            return "Model needs improvement. Consider feature engineering or alternative algorithms."
        else:
            return "Model quality is low. Recommend data quality review and algorithm reassessment."

    # ========== REST OF THE METHODS (same as before) ==========

    def analyze_clusters(self, X_original, cluster_labels, building_ids):
        """Deep analysis of cluster characteristics"""
        print("\n" + "=" * 70)
        print("PHASE 7: CLUSTER CHARACTERIZATION")
        print("=" * 70)

        # Add cluster labels to original features
        analysis_df = X_original.copy()
        analysis_df['Cluster'] = cluster_labels

        # Handle building IDs
        id_columns = ['LCLid', 'building_id', 'id', 'meter_id', 'customer_id']
        id_col = None
        for col in id_columns:
            if col in building_ids.name or any(c in str(building_ids.name) for c in id_columns):
                id_col = building_ids.name
                break

        analysis_df[id_col if id_col else 'building_id'] = building_ids.values

        # Cluster profiles (exclude ID from mean)
        cluster_profiles = analysis_df.drop([id_col if id_col else 'building_id'], axis=1).groupby('Cluster').mean()

        # Statistical significance testing (ANOVA)
        print("\n" + "-" * 70)
        print("STATISTICAL SIGNIFICANCE (ANOVA F-statistic):")
        print("-" * 70)

        significant_features = []
        for feature in self.feature_names:
            groups = [analysis_df[analysis_df['Cluster'] == i][feature].values
                     for i in range(len(cluster_profiles))]
            f_stat, p_value = stats.f_oneway(*groups)

            if p_value < 0.05:
                significant_features.append((feature, f_stat, p_value))

        # Sort by F-statistic
        significant_features.sort(key=lambda x: x[1], reverse=True)

        print("\nTop 10 Most Discriminative Features:")
        for feature, f_stat, p_value in significant_features[:10]:
            print(f"  {feature}: F={f_stat:.2f}, p={p_value:.2e}")

        # Cluster naming based on characteristics
        print("\n" + "-" * 70)
        print("CLUSTER PROFILES:")
        print("-" * 70)

        cluster_names = []
        for cluster_id in range(len(cluster_profiles)):
            profile = cluster_profiles.loc[cluster_id]

            # Characterization logic
            if profile['peak_to_avg_ratio'] > cluster_profiles['peak_to_avg_ratio'].median():
                if profile['coefficient_of_variation'] > cluster_profiles['coefficient_of_variation'].median():
                    name = "High Peak & Variable"
                else:
                    name = "Peak-Intensive Stable"
            else:
                if profile['load_factor'] > cluster_profiles['load_factor'].median():
                    name = "Efficient Steady"
                elif profile['weekday_weekend_ratio'] < 0.8:
                    name = "Weekend-Heavy"
                else:
                    name = "Baseline Consistent"

            cluster_names.append(name)

            count = (cluster_labels == cluster_id).sum()
            print(f"\nCluster {cluster_id}: '{name}' ({count} buildings)")
            print(f"  Key Characteristics:")
            print(f"    - Avg Consumption: {profile['mean_consumption']:.3f} kWh")
            print(f"    - Peak/Avg Ratio: {profile['peak_to_avg_ratio']:.2f}x")
            print(f"    - Load Factor: {profile['load_factor']:.2%}")
            print(f"    - Variability (CV): {profile['coefficient_of_variation']:.2f}")
            print(f"    - Weekday/Weekend: {profile['weekday_weekend_ratio']:.2f}")

        return analysis_df, cluster_profiles, cluster_names

    def generate_recommendations(self, cluster_profiles, cluster_names):
        """Generate targeted efficiency recommendations"""
        print("\n" + "=" * 70)
        print("PHASE 8: OPTIMIZATION RECOMMENDATIONS")
        print("=" * 70)

        recommendations = {}

        for cluster_id, name in enumerate(cluster_names):
            profile = cluster_profiles.loc[cluster_id]
            recs = []

            # Peak demand strategies
            if profile['peak_to_avg_ratio'] > 2.5:
                recs.append("⚡ PEAK SHAVING: Implement demand response programs")
                recs.append("⚡ LOAD SHIFTING: Move non-critical loads to off-peak hours")
                recs.append("⚡ ENERGY STORAGE: Install battery systems for peak reduction")

            # Variability strategies
            if profile['coefficient_of_variation'] > 0.5:
                recs.append("📊 PREDICTIVE CONTROL: Use AI-based HVAC scheduling")
                recs.append("📊 OCCUPANCY SENSORS: Deploy smart occupancy-based controls")
                recs.append("📊 EQUIPMENT AUDIT: Review and optimize equipment runtime")

            # Load factor improvement
            if profile['load_factor'] < 0.6:
                recs.append("⚙️ BASELOAD OPTIMIZATION: Reduce unnecessary constant loads")
                recs.append("⚙️ SCHEDULING: Implement aggressive setback schedules")

            # Seasonal efficiency
            if profile['winter_summer_ratio'] > 1.5:
                recs.append("🌡️ HVAC TUNING: Optimize heating system efficiency")
                recs.append("🌡️ INSULATION: Improve building envelope performance")
            elif profile['winter_summer_ratio'] < 0.7:
                recs.append("❄️ COOLING OPTIMIZATION: Tune cooling systems and controls")
                recs.append("❄️ ECONOMIZER: Implement free cooling strategies")

            # Weekend patterns
            if profile['weekday_weekend_ratio'] > 1.3:
                recs.append("📅 WEEKEND SETBACK: Implement deep weekend energy reductions")

            recommendations[cluster_id] = recs

            print(f"\n{'='*70}")
            print(f"CLUSTER {cluster_id}: {name}")
            print(f"{'='*70}")
            for i, rec in enumerate(recs, 1):
                print(f"{i}. {rec}")

        return recommendations

    def create_visualizations(self, X_pca, cluster_labels, cluster_metrics,
                            cluster_profiles, cluster_names):
        """Generate comprehensive visualizations"""
        print("\n" + "=" * 70)
        print("PHASE 9: CREATING VISUALIZATIONS")
        print("=" * 70)

        fig = plt.figure(figsize=(20, 12))

        # 1. Scree Plot
        ax1 = plt.subplot(3, 4, 1)
        variance_ratio = self.pca.explained_variance_ratio_
        cumsum_variance = np.cumsum(variance_ratio)
        ax1.plot(range(1, len(variance_ratio) + 1), cumsum_variance, 'bo-', linewidth=2)
        ax1.axhline(y=0.90, color='r', linestyle='--', label='90% threshold')
        ax1.set_xlabel('Number of Components')
        ax1.set_ylabel('Cumulative Explained Variance')
        ax1.set_title('PCA Scree Plot')
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # 2. Elbow Method
        ax2 = plt.subplot(3, 4, 2)
        ax2.plot(cluster_metrics['k_range'], cluster_metrics['inertias'], 'go-', linewidth=2)
        ax2.set_xlabel('Number of Clusters (k)')
        ax2.set_ylabel('Inertia (WCSS)')
        ax2.set_title('Elbow Method for Optimal K')
        ax2.grid(True, alpha=0.3)

        # 3. Silhouette Scores
        ax3 = plt.subplot(3, 4, 3)
        ax3.plot(cluster_metrics['k_range'], cluster_metrics['silhouette'], 'mo-', linewidth=2)
        ax3.set_xlabel('Number of Clusters (k)')
        ax3.set_ylabel('Silhouette Score')
        ax3.set_title('Silhouette Analysis')
        ax3.grid(True, alpha=0.3)

        # 4. Calinski-Harabasz Score
        ax4 = plt.subplot(3, 4, 4)
        ax4.plot(cluster_metrics['k_range'], cluster_metrics['calinski_harabasz'], 'co-', linewidth=2)
        ax4.set_xlabel('Number of Clusters (k)')
        ax4.set_ylabel('Calinski-Harabasz Score')
        ax4.set_title('Calinski-Harabasz Score (Higher is Better)')
        ax4.grid(True, alpha=0.3)

        # 5. PCA 2D Scatter
        ax5 = plt.subplot(3, 4, 5)
        scatter = ax5.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels,
                            cmap='viridis', s=50, alpha=0.6, edgecolors='black', linewidth=0.5)
        ax5.set_xlabel(f'PC1 ({variance_ratio[0]:.1%} variance)')
        ax5.set_ylabel(f'PC2 ({variance_ratio[1]:.1%} variance)')
        ax5.set_title('Clusters in PCA Space (PC1 vs PC2)')
        plt.colorbar(scatter, ax=ax5, label='Cluster')

        # 6. PCA 3D Scatter (if available)
        if X_pca.shape[1] >= 3:
            ax6 = plt.subplot(3, 4, 6, projection='3d')
            scatter3d = ax6.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],
                                   c=cluster_labels, cmap='viridis', s=30, alpha=0.6)
            ax6.set_xlabel(f'PC1')
            ax6.set_ylabel(f'PC2')
            ax6.set_zlabel(f'PC3')
            ax6.set_title('3D Cluster Visualization')

        # 7. Cluster Size Distribution
        ax7 = plt.subplot(3, 4, 7)
        unique, counts = np.unique(cluster_labels, return_counts=True)
        colors = plt.cm.viridis(np.linspace(0, 1, len(unique)))
        bars = ax7.bar([cluster_names[i] for i in unique], counts, color=colors, edgecolor='black')
        ax7.set_xlabel('Cluster')
        ax7.set_ylabel('Number of Buildings')
        ax7.set_title('Cluster Size Distribution')
        plt.setp(ax7.xaxis.get_majorticklabels(), rotation=45, ha='right')

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax7.text(bar.get_x() + bar.get_width()/2., height,
                    f'{int(height)}', ha='center', va='bottom')

        # 8. Heatmap of Cluster Centroids
        ax8 = plt.subplot(3, 4, 8)
        top_features = ['mean_consumption', 'peak_to_avg_ratio', 'load_factor',
                       'coefficient_of_variation', 'weekday_weekend_ratio']
        heatmap_data = cluster_profiles[top_features].T

        sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn_r',
                   ax=ax8, cbar_kws={'label': 'Value'}, linewidths=0.5)
        ax8.set_xlabel('Cluster')
        ax8.set_ylabel('Feature')
        ax8.set_title('Cluster Centroids Heatmap')
        ax8.set_xticklabels([cluster_names[i] for i in range(len(cluster_names))], rotation=45, ha='right')

        # 9-12. Accuracy Metrics Visualization
        ax9 = plt.subplot(3, 4, 9)
        metrics_names = ['Silhouette\nScore', 'Davies-\nBouldin', 'Calinski-\nHarabasz']
        metrics_values = [
            self.accuracy_metrics['silhouette_score'],
            1 / (1 + self.accuracy_metrics['davies_bouldin_index']),  # Invert for visualization
            min(self.accuracy_metrics['calinski_harabasz_score'] / 1000, 1.0)  # Normalize
        ]
        bars = ax9.bar(metrics_names, metrics_values, color=['green', 'blue', 'orange'], edgecolor='black')
        ax9.set_ylabel('Normalized Score')
        ax9.set_title('Clustering Quality Metrics')
        ax9.set_ylim([0, 1])
        ax9.grid(True, alpha=0.3, axis='y')

        # Add value labels
        for bar, val in zip(bars, metrics_values):
            height = bar.get_height()
            ax9.text(bar.get_x() + bar.get_width()/2., height,
                    f'{val:.3f}', ha='center', va='bottom')

        # 10. PCA Variance Explained
        ax10 = plt.subplot(3, 4, 10)
        ax10.bar(range(1, len(variance_ratio) + 1), variance_ratio, color='steelblue', edgecolor='black')
        ax10.set_xlabel('Principal Component')
        ax10.set_ylabel('Variance Explained')
        ax10.set_title('Variance Explained by Each PC')
        ax10.grid(True, alpha=0.3, axis='y')

        # 11. Overall Quality Score
        ax11 = plt.subplot(3, 4, 11)
        quality_score = self._calculate_overall_quality()
        wedges, texts, autotexts = ax11.pie([quality_score, 100-quality_score],
                                            labels=['Quality Score', 'Room for Improvement'],
                                            autopct='%1.1f%%',
                                            startangle=90,
                                            colors=['#4CAF50', '#E0E0E0'])
        ax11.set_title(f'Overall Model Quality\n{quality_score:.1f}/100')

        # 12. Cluster Balance
        ax12 = plt.subplot(3, 4, 12)
        cluster_sizes = [self.accuracy_metrics['cluster_sizes'][i] for i in sorted(self.accuracy_metrics['cluster_sizes'].keys())]
        ax12.pie(cluster_sizes, labels=[f'Cluster {i}' for i in sorted(self.accuracy_metrics['cluster_sizes'].keys())],
                autopct='%1.1f%%', startangle=90)
        ax12.set_title('Cluster Size Distribution')

        # plt.tight_layout()
        # plt.savefig('/home/claude/energy_analysis_results_enhanced.png', dpi=300, bbox_inches='tight')
        # print("\n✓ Visualization saved: energy_analysis_results_enhanced.png")

        return fig


# ========== MAIN EXECUTION ==========

def main():
    """Execute complete energy analysis pipeline with custom dataset input"""

    print("\n")
    print("╔" + "=" * 68 + "╗")
    print("║  GOOGLE SUSTAINABILITY ANALYTICS - ENHANCED VERSION            ║")
    print("║  Energy Consumption Pattern Analysis                          ║")
    print("║  PCA + K-Means Clustering with Accuracy Metrics               ║")
    print("╚" + "=" * 68 + "╝")
    print("\n")

    # Initialize pipeline
    pipeline = EnergyAnalysisPipeline()

    # Phase 1: Load Data (with custom input option)
    energy_data, households, weather = pipeline.load_and_merge_data()

    # Phase 2: Feature Engineering
    features_df = pipeline.engineer_features(energy_data, weather)

    # Phase 3: Preprocessing
    X_scaled, building_ids, X_original = pipeline.preprocess_data(features_df)

    # Phase 4: PCA
    X_pca, components_df = pipeline.perform_pca(X_scaled)

    # Phase 5: Determine Optimal Clusters
    optimal_k, cluster_metrics = pipeline.determine_optimal_clusters(X_pca)

    # Phase 6: Apply K-Means
    cluster_labels = pipeline.apply_kmeans(X_pca, optimal_k)

    # ========== PRINT COMPREHENSIVE ACCURACY REPORT ==========
    # pipeline.print_accuracy_report()

    # Phase 7: Analyze Clusters
    analysis_df, cluster_profiles, cluster_names = pipeline.analyze_clusters(
        X_original, cluster_labels, building_ids
    )

    # Phase 8: Generate Recommendations
    recommendations = pipeline.generate_recommendations(cluster_profiles, cluster_names)

    # Phase 9: Create Visualizations
    fig = pipeline.create_visualizations(
        X_pca, cluster_labels, cluster_metrics,
        cluster_profiles, cluster_names
    )


    # Summary Report
    print("\n" + "=" * 70)
    print("ANALYSIS COMPLETE - SUMMARY")
    print("=" * 70)
    print(f"✓ Analyzed {len(building_ids)} buildings")
    print(f"✓ Engineered {len(pipeline.feature_names)} features")
    print(f"✓ Reduced to {X_pca.shape[1]} principal components ({pipeline.accuracy_metrics['pca_variance_retained']*100:.1f}% variance)")
    print(f"✓ Identified {optimal_k} distinct building clusters")
    print(f"✓ Overall Model Quality: {pipeline._calculate_overall_quality():.1f}/100")
    print(f"✓ Silhouette Score: {pipeline.accuracy_metrics['silhouette_score']:.4f}")
    print(f"✓ Davies-Bouldin Index: {pipeline.accuracy_metrics['davies_bouldin_index']:.4f}")
    print(f"✓ Generated targeted recommendations for each cluster")
    print("\n" + "=" * 70)

    return pipeline, analysis_df, cluster_profiles, recommendations



if __name__ == "__main__":
    # Run with interactive mode or default
    import sys

    if len(sys.argv) > 1 and sys.argv[1] == '--default':
        print("Running in default mode (non-interactive)...")
        pipeline = EnergyAnalysisPipeline()
        pipeline, analysis_df, cluster_profiles, recommendations = main()
    else:
        pipeline, analysis_df, cluster_profiles, recommendations = main()



╔====================================================================╗
║  GOOGLE SUSTAINABILITY ANALYTICS - ENHANCED VERSION            ║
║  Energy Consumption Pattern Analysis                          ║
║  PCA + K-Means Clustering with Accuracy Metrics               ║
╚====================================================================╝



PHASE 1: DATA LOADING & PREPARATION
